# SQLite Preprocessing

```
$ sqlite3

> ATTACH 'mxm_dataset.db' AS mxm;

> ATTACH DATABASE 'subset.db' AS subset;

> CREATE TABLE subset.words AS
> SELECT * FROM mxm.words;

> CREATE temp TABLE temp_tracks AS
> SELECT DISTINCT track_id FROM mxm.lyrics LIMIT 1000;

> CREATE TABLE subset.lyrics AS
> SELECT * FROM mxm.lyrics
> WHERE track_id IN (SELECT track_id FROM temp_tracks);

> DETACH DATABASE subset;
> .quit
```
```
```

# TF-IDF

## Load data

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('subset.db')

lyrics_df = pd.read_sql_query('SELECT track_id, word, count FROM lyrics', conn)

words_df = pd.read_sql_query('SELECT word FROM words', conn)

song_count = pd.read_sql_query('SELECT COUNT(DISTINCT track_id) FROM lyrics', conn)

print(f'{song_count.iloc[0,0]}')
print('Total number of words:', len(lyrics_df))
display(lyrics_df.head())

print(f'Top {len(words_df)} most useful words.')
display(words_df.head())

## Pivot Table => Bag of Words

In [42]:
%pip install nltk

  Using cached nltk-3.9.1-py3-none-any.whl (1.5 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.7/781.7 kB 21.1 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


In [322]:
import nltk
from nltk.corpus import words, stopwords
from nltk.stem import PorterStemmer

nltk.download('words')
nltk.download('stopwords')

ps = PorterStemmer()

english_vocab = set(ps.stem(w.lower()) for w in words.words())
stop_words = set(w.lower() for w in stopwords.words('english'))

top_words = set(words_df['word'])
valid_words = {w for w in top_words if w in english_vocab and w not in stop_words and len(w) >= 4}

filtered_lyrics_df = lyrics_df[lyrics_df['word'].isin(valid_words)]

bow_df = filtered_lyrics_df.pivot_table(index='track_id', columns='word', values='count', fill_value=0)

n_english_words = (bow_df > 0).sum(axis=1)

total_words = lyrics_df.groupby('track_id').size()
n_english_words = n_english_words.reindex(total_words.index).fillna(0)

proportion_english = n_english_words / total_words

threshold = 0.65
min_english_words = 3
tracks_to_keep = (proportion_english >= threshold) & (n_english_words >= min_english_words)

bow_df = bow_df.loc[tracks_to_keep]

[nltk_data] Downloading package words to /home/jovyan/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [323]:
from sklearn.feature_extraction.text import TfidfTransformer

tfidf = TfidfTransformer()

tfidf_matrix = tfidf.fit_transform(bow_df)

In [324]:
tfidf_matrix

<94x2963 sparse matrix of type '<class 'numpy.float64'>'
	with 6086 stored elements in Compressed Sparse Row format>

In [327]:
words = bow_df.columns
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), index=bow_df.index, columns=words)

In [328]:
tfidf_df.head()

word,abandon,aboard,abov,absenc,absolut,absurd,abus,abyss,accept,accid,...,yellow,yesterday,yonder,york,young,younger,youth,zero,zombi,zone
track_id,,,,,,,,,,,,,,,,,,,,,
TRAABEV12903CC53A4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0
TRAACZN128F93236B1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.250551,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0
TRAAKMX128F934B494,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0
TRAAQJO128F92E9DD0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.000000,0.0,0.0,0.0,0.15008,0.0,0.0,0.0,0.0,0.0
TRAAVEY128F9342A2B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.121165,0.0,...,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0


In [330]:
from sklearn.cluster import KMeans

n = 2
kmeans = KMeans(n_clusters=n, random_state=1)
kmeans.fit(tfidf_matrix)

labels = kmeans.labels_

In [331]:
clusters = {}
valid_clusters_length = len(set(labels)) - (1 if -1 in labels else 0)
for i in range(valid_clusters_length):
    print(f'\nCluster {i}:')
    clusters[i] = bow_df[labels == i].index.tolist()
    print(f'Cluster length: {len(clusters[i])}')
    print(clusters[i][:10])


Cluster 0:
Cluster length: 48
['TRAABEV12903CC53A4', 'TRAACZN128F93236B1', 'TRAAKMX128F934B494', 'TRAAQJO128F92E9DD0', 'TRAAVEY128F9342A2B', 'TRABCPG128F42466F3', 'TRACBPX128F9342ABA', 'TRACHLU128F4259F9B', 'TRACJJC128F934AF55', 'TRACKWV128F934B6B0']

Cluster 1:
Cluster length: 46
['TRABDTV128E0791839', 'TRABIVC128F934B524', 'TRABKJV128F92E9965', 'TRABTWX128F42B8619', 'TRADJBU128F42951F1', 'TRADPEV128F4257A51', 'TRADQQD12903CB0695', 'TRADROH128F932B551', 'TRADZZP128F428BCE6', 'TRAEEPX128F92E2464']


In [332]:
def get_words(track_id):
    df = pd.read_sql_query(
        f'SELECT track_id, word, count FROM lyrics WHERE track_id="{track_id}"', conn
    )

    return ', '.join(df.groupby('track_id')['word'].agg(list).iloc[0])

def get_words_in_cluster(cluster_ids):
    results = {}
    for i in cluster_ids:
        results[i] = get_words(i)

    return results

In [333]:
import json
from collections import Counter
from nltk.corpus import stopwords

import nltk
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def get_top_words_in_cluster(track_ids, bow_df, top_n=10):
    cluster_df = bow_df.loc[track_ids]

    word_counts = cluster_df.sum(axis=0)

    word_count_dict = word_counts.to_dict()

    filtered_word_counts = {word: count for word, count in word_count_dict.items() if word not in stop_words}

    sorted_words = sorted(filtered_word_counts.items(), key=lambda x: x[1], reverse=True)

    top_words = [word for word, count in sorted_words[:top_n]]

    return top_words

for i in range(0, valid_clusters_length):
    print(f'\nCluster {i}:')

    top_words = get_top_words_in_cluster(clusters[i], bow_df, top_n=10)

    print(json.dumps(top_words, indent=4))


Cluster 0:
[
    "world",
    "mind",
    "blood",
    "hell",
    "death",
    "soul",
    "burn",
    "life",
    "like",
    "time"
]

Cluster 1:
[
    "babi",
    "night",
    "dark",
    "life",
    "like",
    "dream",
    "time",
    "love",
    "soul",
    "cold"
]


[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
